In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

from langchain_chroma import Chroma

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6942.98it/s]


In [5]:
from langchain_core.documents import Document

doc1 = Document(
    page_content="Virat Kohli is one of India's greatest batsmen. He has scored over 14,000 ODI runs and is known for his consistency and aggressive batting.",
    metadata={"ipl_team": "Royal Challengers Bengaluru"}
)

doc2 = Document(
    page_content="Rohit Sharma is the captain of India's ODI team. He is the only batsman to score three double centuries in One Day Internationals.",
    metadata={"ipl_team": "Mumbai Indians"}
)

doc3 = Document(
    page_content="Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all formats.",
    metadata={"ipl_team": "Mumbai Indians"}
)

doc4 = Document(
    page_content="Ravindra Jadeja is one of the world's best all-rounders. He contributes with batting, left-arm spin bowling, and outstanding fielding.",
    metadata={"ipl_team": "Chennai Super Kings"}
)

# Combine them into a list when needed
docs = [doc1, doc2, doc3, doc4]

In [6]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [7]:
# add documents in created vector store
vector_store.add_documents(docs)

['d82decc4-5a87-4774-a704-fedd29487122',
 '676ad0f0-a9d3-4c90-92ee-5c2220a211d2',
 '92b58cff-b7dd-4a2b-a2a6-47ad343a9394',
 'dd260f59-c2e3-4410-b839-197b3244c57f']

In [9]:
vector_store.get(include=['embeddings','metadatas','documents'])

{'ids': ['d82decc4-5a87-4774-a704-fedd29487122',
  '676ad0f0-a9d3-4c90-92ee-5c2220a211d2',
  '92b58cff-b7dd-4a2b-a2a6-47ad343a9394',
  'dd260f59-c2e3-4410-b839-197b3244c57f'],
 'embeddings': array([[ 0.0629332 ,  0.06813792, -0.08417869, ..., -0.01742304,
          0.04391187, -0.0195422 ],
        [ 0.03410647,  0.01033769, -0.04815709, ..., -0.01592463,
          0.00276925, -0.04490923],
        [ 0.00865926, -0.02012677, -0.05325401, ..., -0.09739757,
          0.012657  ,  0.08300489],
        [ 0.02069626,  0.01976965, -0.04522886, ..., -0.10294682,
         -0.02769829,  0.03958848]], shape=(4, 384)),
 'documents': ["Virat Kohli is one of India's greatest batsmen. He has scored over 14,000 ODI runs and is known for his consistency and aggressive batting.",
  "Rohit Sharma is the captain of India's ODI team. He is the only batsman to score three double centuries in One Day Internationals.",
  "Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling acti

In [11]:
# seaching documents

vector_store.similarity_search(
    query='Who among these are a bowler',
    k=1
)

[Document(id='92b58cff-b7dd-4a2b-a2a6-47ad343a9394', metadata={'ipl_team': 'Mumbai Indians'}, page_content="Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all formats.")]

In [13]:
# searching with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler',
    k=2
)

[(Document(id='92b58cff-b7dd-4a2b-a2a6-47ad343a9394', metadata={'ipl_team': 'Mumbai Indians'}, page_content="Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all formats."),
  1.0629494190216064),
 (Document(id='dd260f59-c2e3-4410-b839-197b3244c57f', metadata={'ipl_team': 'Chennai Super Kings'}, page_content="Ravindra Jadeja is one of the world's best all-rounders. He contributes with batting, left-arm spin bowling, and outstanding fielding."),
  1.168199896812439)]

In [16]:
# meta data filtering
vector_store.similarity_search_with_score(
    query='Which among these is bowler?',
    filter={'ipl_team': 'Mumbai Indians'}
)

[(Document(id='92b58cff-b7dd-4a2b-a2a6-47ad343a9394', metadata={'ipl_team': 'Mumbai Indians'}, page_content="Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all formats."),
  0.9513432383537292),
 (Document(id='676ad0f0-a9d3-4c90-92ee-5c2220a211d2', metadata={'ipl_team': 'Mumbai Indians'}, page_content="Rohit Sharma is the captain of India's ODI team. He is the only batsman to score three double centuries in One Day Internationals."),
  1.1640312671661377)]

In [21]:
# updating an existing document
update_doc1=Document(
    page_content="Virat Kohili is former captain of royal challenger bangalore (RCB). He is a great batsman with aggresion",
    metadata={'ipl_team':'Royal Challengers Bangalore'}
)

vector_store.update_document(document_id='d82decc4-5a87-4774-a704-fedd29487122',document=update_doc1)

In [22]:
vector_store.get(include=['embeddings','metadatas','documents'])

{'ids': ['d82decc4-5a87-4774-a704-fedd29487122',
  '676ad0f0-a9d3-4c90-92ee-5c2220a211d2',
  '92b58cff-b7dd-4a2b-a2a6-47ad343a9394',
  'dd260f59-c2e3-4410-b839-197b3244c57f'],
 'embeddings': array([[-0.03058343,  0.08406352, -0.03498945, ..., -0.05622405,
          0.02238437, -0.01771167],
        [ 0.03410647,  0.01033769, -0.04815709, ..., -0.01592463,
          0.00276925, -0.04490923],
        [ 0.00865926, -0.02012677, -0.05325401, ..., -0.09739757,
          0.012657  ,  0.08300489],
        [ 0.02069626,  0.01976965, -0.04522886, ..., -0.10294682,
         -0.02769829,  0.03958848]], shape=(4, 384)),
 'documents': ['Virat Kohili is former captain of royal challenger bangalore (RCB). He is a great batsman with aggresion',
  "Rohit Sharma is the captain of India's ODI team. He is the only batsman to score three double centuries in One Day Internationals.",
  "Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all f

In [23]:
# deleting a particular document from vectore store

vector_store.delete(ids=['d82decc4-5a87-4774-a704-fedd29487122'])
vector_store.get(include=['embeddings','metadatas','documents'])

{'ids': ['676ad0f0-a9d3-4c90-92ee-5c2220a211d2',
  '92b58cff-b7dd-4a2b-a2a6-47ad343a9394',
  'dd260f59-c2e3-4410-b839-197b3244c57f'],
 'embeddings': array([[ 0.03410647,  0.01033769, -0.04815709, ..., -0.01592463,
          0.00276925, -0.04490923],
        [ 0.00865926, -0.02012677, -0.05325401, ..., -0.09739757,
          0.012657  ,  0.08300489],
        [ 0.02069626,  0.01976965, -0.04522886, ..., -0.10294682,
         -0.02769829,  0.03958848]], shape=(3, 384)),
 'documents': ["Rohit Sharma is the captain of India's ODI team. He is the only batsman to score three double centuries in One Day Internationals.",
  "Jasprit Bumrah is India's premier fast bowler. He is famous for his unique bowling action and exceptional yorkers in all formats.",
  "Ravindra Jadeja is one of the world's best all-rounders. He contributes with batting, left-arm spin bowling, and outstanding fielding."],
 'uris': None,
 'included': ['embeddings', 'metadatas', 'documents'],
 'data': None,
 'metadatas': [{'i